# 関連銘柄データ取得ツール

In [ ]:
import sqlite3
import pandas as pd
import os
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# 設定
DB_PATH = '../data/stock_data.db'
DICT_PATH = '../data/ticker_dictionary.json'
ESTIMATED_KB_PER_STOCK = 20 # 1銘柄あたりの推定ファイルサイズ(KB)

out = widgets.Output()

def get_related_stocks(code):
    if not os.path.exists(DICT_PATH):
        print(f"Error: Dictionary not found at {DICT_PATH}")
        return []
    
    with open(DICT_PATH, 'r', encoding='utf-8') as f:
        ticker_dict = json.load(f)
        
    # 入力された銘柄の属性を取得
    target_info = None
    for item in ticker_dict:
        if item.get('code') == code:
            target_info = item
            break
            
    if not target_info:
        print(f"銘柄コード {code} が辞書に見つかりません。")
        return [code] # 辞書にない場合は入力された銘柄のみ
        
    layered_tags = target_info.get('layered_tags', {})
    target_sectors = set(layered_tags.get('Sector_Major', []))
    target_techs = set(layered_tags.get('Core_Product_Tech', []))
    target_industry = target_info.get('industry')
    
    related_codes = set()
    for item in ticker_dict:
        item_layered = item.get('layered_tags', {})
        item_sectors = set(item_layered.get('Sector_Major', []))
        item_techs = set(item_layered.get('Core_Product_Tech', []))
        item_industry = item.get('industry')
        
        # Sector_Major, Core_Product_Tech のいずれかのタグが共通、または industry が一致するものを関連銘柄とする
        if (target_sectors & item_sectors) or (target_techs & item_techs) or (target_industry and target_industry == item_industry):
            if item.get('code'):
                related_codes.add(item.get('code'))
                
    # 確実に入力コードは含める
    related_codes.add(code)
    
    return list(related_codes)

def fetch_single_stock_data(code, conn):
    time_series_tables = {
        'daily_prices': 'date',
        'daily_financials': 'date',
        'weekly_margin': 'date',
        'financial_results': 'date',
        'stock_daily_indicators': 'date',
        'daily_trade_indicators': 'date',
        'stock_eps_revisions': 'revision_date',
        'corporate_disclosures': 'disclosure_date'
    }
    
    dfs = []
    
    for table, date_col in time_series_tables.items():
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table,))
        if not cursor.fetchone():
            continue
            
        query = f"SELECT * FROM {table} WHERE code = ?"
        df = pd.read_sql_query(query, conn, params=(code,))
        
        if df.empty:
            continue
            
        if table == 'corporate_disclosures':
            cols_to_drop = ['ai_summary', 'title', 'document_url', 'created_at', 'pdf_text']
            df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
            
        if date_col != 'date' and date_col in df.columns:
            df = df.rename(columns={date_col: 'date'})
            
        if 'date' in df.columns:
            df = df.sort_values('date').reset_index(drop=True)
            
            numeric_cols = df.select_dtypes(include='number').columns.tolist()
            exclude_from_diff = ['id', 'code', 'date']
            numeric_cols = [c for c in numeric_cols if c not in exclude_from_diff]
            
            diff_1d = df[numeric_cols].diff(1).add_suffix('_1d_diff')
            diff_1w = df[numeric_cols].diff(5).add_suffix('_1w_diff')
            diff_1m = df[numeric_cols].diff(20).add_suffix('_1m_diff')
            
            df = pd.concat([df, diff_1d, diff_1w, diff_1m], axis=1)
            
            rename_dict = {c: f"{table}_{c}" for c in df.columns if c not in ['code', 'date']}
            df = df.rename(columns=rename_dict)
            
            dfs.append(df)
            
    if not dfs:
        return pd.DataFrame()

    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=['code', 'date'], how='outer')
        
    merged_df = merged_df.sort_values('date').reset_index(drop=True)
    
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='companies'")
    if cursor.fetchone():
        comp_df = pd.read_sql_query("SELECT * FROM companies WHERE code = ?", conn, params=(code,))
        if not comp_df.empty:
            comp_rename = {c: f"companies_{c}" for c in comp_df.columns if c != 'code'}
            comp_df = comp_df.rename(columns=comp_rename)
            merged_df = pd.merge(merged_df, comp_df, on='code', how='left')
            
    return merged_df

def fetch_and_save_data(related_codes, output_path):
    if not os.path.exists(DB_PATH):
        print(f"Error: Database not found at {DB_PATH}")
        return

    conn = sqlite3.connect(DB_PATH)
    all_dfs = []
    
    total = len(related_codes)
    for i, code in enumerate(related_codes):
        print(f"処理中: {code} ({i+1}/{total})")
        df = fetch_single_stock_data(code, conn)
        if not df.empty:
            all_dfs.append(df)
            
    conn.close()
    
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\n完了！データを保存しました: {output_path}")
    else:
        print("\n取得できるデータがありませんでした。")

# GUI要素の作成
code_input = widgets.Text(
    value='',
    placeholder='例: 1301',
    description='銘柄コード:',
    disabled=False
)

check_btn = widgets.Button(
    description='対象銘柄を確認',
    button_style='info',
    tooltip='関連銘柄の数と推定ファイルサイズを確認します'
)

save_btn = widgets.Button(
    description='データを保存',
    button_style='success',
    tooltip='CSVファイルとしてデータを取得・保存します',
    disabled=True
)

current_related_codes = []

def on_check_clicked(b):
    with out:
        clear_output()
        code = code_input.value.strip()
        if not code:
            print("銘柄コードを入力してください。")
            return
            
        global current_related_codes
        current_related_codes = get_related_stocks(code)
        
        count = len(current_related_codes)
        est_size = count * ESTIMATED_KB_PER_STOCK
        
        print(f"対象銘柄コード: {code}")
        print(f"関連銘柄数: {count} 銘柄")
        print(f"推定ファイルサイズ: 約 {est_size} KB")
        print("------------------------")
        print("この対象銘柄でデータを保存しますか？")
        
        save_btn.disabled = False

def on_save_clicked(b):
    with out:
        clear_output()
        save_btn.disabled = True
        if not current_related_codes:
            print("先に対象銘柄を確認してください。")
            return
            
        code = code_input.value.strip()
        output_path = f"stock_{code}_related_data.csv"
        print(f"データ取得を開始します...対象 {len(current_related_codes)} 銘柄")
        fetch_and_save_data(current_related_codes, output_path)

check_btn.on_click(on_check_clicked)
save_btn.on_click(on_save_clicked)

display(widgets.HBox([code_input, check_btn, save_btn]), out)
